In [1]:
import glob
import scipy.io as sio
import pandas as pd
import re
import os
from matplotlib import pyplot as plt

In [2]:
data_path = glob.glob(os.path.join('..', 'data', 'Batter', '*', '*.mat'))
target_path = glob.glob(os.path.join('..', 'data', 'Batter', '*', '*.xlsx'))

In [3]:
from collections import defaultdict
import os

# 创建一个默认字典
parent_folder_dict = defaultdict(list)

# 遍历 target_path，将父文件夹作为键，路径作为值
for path in target_path:
    parent_folder = os.path.basename(os.path.dirname(path))
    parent_folder_dict[parent_folder].append(path)

# 转换为普通字典（可选）
parent_folder_dict = dict(parent_folder_dict)
parent_folder_dict

{'NMC#3_charge': ['../data/Batter/NMC#3_charge/相位数据对应到容量值.xlsx'],
 'LCO#4_charge': ['../data/Batter/LCO#4_charge/相位数据对应到容量值.xlsx'],
 'NMC#1_discharge': ['../data/Batter/NMC#1_discharge/相位数据对应到容量值.xlsx'],
 'LCO#2_charge': ['../data/Batter/LCO#2_charge/相位数据对应到容量值.xlsx'],
 'LCO#3_charge': ['../data/Batter/LCO#3_charge/相位数据对应到容量值.xlsx'],
 'NMC#1_charge': ['../data/Batter/NMC#1_charge/相位数据对应到容量值.xlsx'],
 'NMC#2_discharge': ['../data/Batter/NMC#2_discharge/相位数据对应到容量值.xlsx'],
 'LCO#4_discharge': ['../data/Batter/LCO#4_discharge/相位数据对应到容量值.xlsx'],
 'NMC#4_discharge': ['../data/Batter/NMC#4_discharge/相位数据对应到容量值.xlsx'],
 'NMC#2_charge': ['../data/Batter/NMC#2_charge/相位数据对应到容量值.xlsx'],
 'NMC#4_charge': ['../data/Batter/NMC#4_charge/相位数据对应到容量值.xlsx'],
 'LCO#3_discharge': ['../data/Batter/LCO#3_discharge/相位数据对应到容量值.xlsx'],
 'NMC#3_discharge': ['../data/Batter/NMC#3_discharge/相位数据对应到容量值.xlsx']}

In [4]:
for key, value in parent_folder_dict.items():
    if 'discharge' in key:
        ta_df = pd.read_excel(value[0], sheet_name='修正后')[['step', '容量(Ah)']]
        parent_folder_dict[key] = ta_df
    else:   
        ta_df = pd.read_excel(value[0])[['step', '容量(Ah)']]
        parent_folder_dict[key] = ta_df

In [5]:
def draw_multi_channel(data, result_path):
    num_channels = data.shape[2]
    cols = 8  # 确保 rows * cols >= num_channels
    rows = (num_channels + cols - 1) // cols  # 向上取整

    fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 2))

    # 将 axes 转换为一维数组，方便索引
    axes = axes.flatten()

    for channel in range(num_channels):
        ax = axes[channel]
        im = ax.imshow(data[:, :, channel])
        ax.set_title(f'Channel {channel}')
        fig.colorbar(im, ax=ax)

    # 隐藏多余的子图
    for ax in axes[num_channels:]:
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(result_path)
    plt.close(fig)  # 关闭图像，避免显示

In [6]:
result_dir = os.path.join('..', 'data', 'visualization')
os.makedirs(result_dir, exist_ok=True)
for batter_name in parent_folder_dict.keys():
    # 读取电池图像数据
    if "NMC#3_discharge" not in batter_name:
        continue
    batter_data = [i for i in data_path if batter_name in i]
    batter_result_path = os.path.join(result_dir, batter_name)
    os.makedirs(batter_result_path, exist_ok=True)
    for batter in batter_data:
        data = sio.loadmat(batter)
        data = data['a']
        vis_file = os.path.join(batter_result_path, os.path.basename(batter).replace('.mat', '.png'))
        draw_multi_channel(data, vis_file)

In [4]:
chans_map = {
    "LCO#4_charge" : (14, 21),
    "LCO#4_discharge": (14, 21),
    "LCO#2_charge" : (14, 21),
    "LCO#3_charge" : (15, 22),
    "LCO#3_discharge" : (15, 22),
    "NMC#1_charge" : (13, 20),
    "NMC#1_discharge" : (15, 22),
    "NMC#2_charge" : (14, 21),
    "NMC#2_discharge" : (15, 22),
    "NMC#3_charge":(10,17),
    "NMC#3_discharge":(10,17),
    "NMC#4_charge" : (15, 22),
    "NMC#4_discharge" : (15, 22),
}

In [15]:
data_dict = dict()
result_path = "../data/Battery_selection"
os.makedirs(result_path, exist_ok=True)
for idx, i in enumerate(data_path):
    batterty_name = os.path.basename(os.path.dirname(i))
    stage = re.search(r'wo_\d+', i).group()
    mah = parent_folder_dict[batterty_name][parent_folder_dict[batterty_name]['step'] == stage]['容量(Ah)'].values[0]
    if "伪影" in i:
        continue
    print(f'Processing {idx+1}/{len(data_path)}, {i}')
    print(batterty_name, stage, mah)
    data = sio.loadmat(i)
    data = data['a']
    
    # data_dict[i] = data
    start_channel, end_channel = chans_map[batterty_name]
    data = data[:,:,start_channel:end_channel]

    target_path = os.path.join(result_path, batterty_name+ '_'+ stage + '.mat')
    sio.savemat(target_path, {'data': data, 'ah': mah})
    
    

Processing 2/114, ../data/Batter/NMC#3_charge/wo_5_#3D.mat
NMC#3_charge wo_5 0.583
Processing 3/114, ../data/Batter/NMC#3_charge/wo_6_#3D.mat
NMC#3_charge wo_6 0.6996
Processing 5/114, ../data/Batter/NMC#3_charge/wo_8_#3D.mat
NMC#3_charge wo_8 0.9328
Processing 7/114, ../data/Batter/NMC#3_charge/wo_7_#3D.mat
NMC#3_charge wo_7 0.8162
Processing 9/114, ../data/Batter/NMC#3_charge/wo_4_#3D.mat
NMC#3_charge wo_4 0.4664
Processing 10/114, ../data/Batter/LCO#4_charge/wo_2_#3D.mat
LCO#4_charge wo_2 0.2332
Processing 11/114, ../data/Batter/LCO#4_charge/wo_5_#3D.mat
LCO#4_charge wo_5 0.583
Processing 12/114, ../data/Batter/LCO#4_charge/wo_6_#3D.mat
LCO#4_charge wo_6 0.6996
Processing 13/114, ../data/Batter/LCO#4_charge/wo_8_#3D.mat
LCO#4_charge wo_8 0.8736
Processing 14/114, ../data/Batter/LCO#4_charge/wo_0_#3D.mat
LCO#4_charge wo_0 0.0
Processing 15/114, ../data/Batter/LCO#4_charge/wo_3_#3D.mat
LCO#4_charge wo_3 0.3498
Processing 16/114, ../data/Batter/LCO#4_charge/wo_7_#3D.mat
LCO#4_charge wo